In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.subscription_checker import check_domain_full

In [2]:
import os
print(os.getcwd())

c:\Users\josen\Desktop\pricing-wayback-scraper\notebooks


In [3]:
import pandas as pd

df = pd.read_csv("../data/news_domain_cleaned_sim.csv")
print(df.shape)
df.describe(include="all")

(299, 5)


,domain,top_domain,traffic,primary_country_code,primary_country_share
count,299,299,2.990000e+02,299,299.000000
unique,299,299,NaN,44,NaN
top,news.google.com,google.com,NaN,US,NaN
freq,1,1,NaN,71,NaN
mean,NaN,NaN,4.659310e+08,NaN,0.768062
std,NaN,NaN,5.495275e+09,NaN,0.238369
min,NaN,NaN,3.546611e+07,NaN,0.092819
25%,NaN,NaN,4.660485e+07,NaN,0.701439
50%,NaN,NaN,6.427469e+07,NaN,0.857779
75%,NaN,NaN,9.642194e+07,NaN,0.939717


In [4]:
df_sorted = df.sort_values(by="traffic", ascending=False) #First rank domains by popularity (more popular = first)
df_sorted.head(20)

,domain,top_domain,traffic,primary_country_code,primary_country_share
0,news.google.com,google.com,94881645895,US,0.192977
1,arz.m.wikipedia.org,wikipedia.org,5143703369,US,0.222355
2,au.sports.yahoo.com,yahoo.com,4495242317,US,0.487774
3,article.yahoo.co.jp,yahoo.co.jp,1846124587,JP,0.982874
4,weather.com,weather.com,1644982011,US,0.417399
5,social.technet.microsoft.com,microsoft.com,992365009,US,0.171637
6,sway.office.com,office.com,873983152,US,0.297375
7,finance.naver.com,naver.com,845945497,KR,0.916839
8,arabic.cnn.com,cnn.com,674632188,US,0.744267
9,myaccount.nytimes.com,nytimes.com,669840885,US,0.685587


In [5]:
import requests #using requests library to access response status codes to help speed up process of checking for paywalls

response = requests.get("https://example.com/subscribe")
print(response.status_code)

404


In [6]:
import requests

response = requests.get("https://myaccount.nytimes.com")
print(response.status_code)
print(response.url)
print(response.history)

403
https://myaccount.nytimes.com/auth/login?response_type=cookie&client_id=acct&redirect_uri=https%3A%2F%2Fmyaccount.nytimes.com%2F
[<Response [302]>]


In [7]:
import requests
from src.subscription_checker import check_domain_full
# Checks a domain for common subscription-related URL paths and classifies
# each response as a positive, negative, or needs-manual-review signal.

def check_domains(domain, paths=["subscribe", "membership", "join", "pricing"]): 
    results = []
    for path in paths:
        url = f"https://{domain}/{path}" 
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                # Exact match: final URL path is identical to what we requested
                # (no redirect elsewhere). Strong signal the page is real.
                if response.url.endswith(f"https://{domain}/{path}"): 
                    verdict = "positive"
                elif any(keyword in response.url for keyword in ["subscri", "member", "plan"]):
                    verdict = "likely positive"
                else:
                    # 200, but the final URL differs from what we requested —
                    # could be a redirect to a real subscribe page (like NYT's
                    # /subscription redirect) OR a bounce to the bare homepage
                    # (like Naver). Can't tell which from status code alone,
                    # so flag for a human to glance at final_url.
                    verdict = "manual positive" 
            elif response.status_code == 404:
                # No page at this path = confident negative, no review needed.
                verdict = "clean negative" 
            else:
                # Any other status code (500, 403, etc.) = behavior unclear,
                # needs manual human look.
                verdict = "manual negative"
            results.append({
    "domain": domain,
    "path": path,
    "status": response.status_code,
    "final_url": response.url,
    "verdict": verdict
})
        except requests.exceptions.RequestException as e:
            # Request itself failed (timeout, DNS error, connection refused)
            # this is the "could not be crawled" case from the task doc,
            # distinct from "no subscription."
            results.append({
        "domain": domain,
        "path": path,
        "status": None,
        "final_url": None,
        "verdict": "error",
        "error": str(e)
        })
    return results

check_domain_full("www.nytimes.com")


[{'domain': 'www.nytimes.com',
  'path': 'subscribe',
  'status': 200,
  'final_url': 'https://www.nytimes.com/subscription?campaignId=9FRJJ',
  'verdict': 'likely positive',
  'price_matches': [{'match': '$1',
    'context': '=="string"&&/^https/.test(t)?t=String(t).replace(/([^/])$/,"$1/"):(f.push("hostname must be a valid URL"),t=null),n!==null'},
   {'match': '$1',
    'context': 'n=="string"&&/^http/.test(n)?n=String(n).replace(/([^/])$/,"$1/"):(f.push("event_gate_hostname must be a valid URL"),n=nul'}]},
 {'domain': 'www.nytimes.com',
  'path': 'membership',
  'status': 404,
  'final_url': 'https://www.nytimes.com/membership',
  'verdict': 'clean negative',
  'price_matches': None},
 {'domain': 'www.nytimes.com',
  'path': 'join',
  'status': 404,
  'final_url': 'https://www.nytimes.com/join',
  'verdict': 'clean negative',
  'price_matches': None},
 {'domain': 'www.nytimes.com',
  'path': 'pricing',
  'status': 404,
  'final_url': 'https://www.nytimes.com/pricing',
  'verdict': 

In [8]:
check_domains("www.nytimes.com")

[{'domain': 'www.nytimes.com',
  'path': 'subscribe',
  'status': 200,
  'final_url': 'https://www.nytimes.com/subscription?campaignId=9FRJJ',
  'verdict': 'likely positive'},
 {'domain': 'www.nytimes.com',
  'path': 'membership',
  'status': 404,
  'final_url': 'https://www.nytimes.com/membership',
  'verdict': 'clean negative'},
 {'domain': 'www.nytimes.com',
  'path': 'join',
  'status': 404,
  'final_url': 'https://www.nytimes.com/join',
  'verdict': 'clean negative'},
 {'domain': 'www.nytimes.com',
  'path': 'pricing',
  'status': 404,
  'final_url': 'https://www.nytimes.com/pricing',
  'verdict': 'clean negative'}]

In [9]:
check_domains("finance.naver.com")

[{'domain': 'finance.naver.com',
  'path': 'subscribe',
  'status': 200,
  'final_url': 'https://finance.naver.com/subscribe',
  'verdict': 'positive'},
 {'domain': 'finance.naver.com',
  'path': 'membership',
  'status': 200,
  'final_url': 'https://finance.naver.com/membership',
  'verdict': 'positive'},
 {'domain': 'finance.naver.com',
  'path': 'join',
  'status': 200,
  'final_url': 'https://finance.naver.com/join',
  'verdict': 'positive'},
 {'domain': 'finance.naver.com',
  'path': 'pricing',
  'status': 200,
  'final_url': 'https://finance.naver.com/pricing',
  'verdict': 'positive'}]

In [10]:
import re

if re.search(r"\$\d+(\.\d{2})?", response.text):
    # price pattern found somewhere on the page
    response = requests.get("https://www.nytimes.com/subscription?campaignId=9FRJJ")
    re.search(r"\$\d+(\.\d{2})?", response.text)

In [11]:
re.search(r"\$\d+(\.\d{2})?", response.text)

In [12]:
response = requests.get("https://www.nytimes.com/subscription?campaignId=9FRJJ")
re.search(r"\$\d+(\.\d{2})?", response.text)


<re.Match object; span=(28921, 28923), match='$1'>

In [14]:
response.text[28850:28950]

'rn typeof t=="string"&&/^https/.test(t)?t=String(t).replace(/([^/])$/,"$1/"):(f.push("hostname must '

In [15]:
# LIMITATION: requests.get() only returns raw HTML/JS source, not what a
# browser actually renders after running JavaScript. Tested against
# nytimes.com/subscription — the only "$" match found was inside a
# JS utility function (a regex backreference "$1", unrelated to pricing),
# not real page content. This suggests pricing on JS-heavy sites is
# rendered client-side and won't be visible to this method.
# Content-based price detection is therefore unreliable for such sites;
# these are routed to manual review rather than trusted automatically.

In [16]:
from src.wayback_client import get_wayback_snapshots

In [18]:
snapshots = get_wayback_snapshots("nytimes.com/subscription")

In [19]:
snapshots

[{'urlkey': 'com,nytimes)/subscription',
  'timestamp': '20210101012345',
  'original': 'https://www.nytimes.com/subscription/',
  'mimetype': 'text/html',
  'statuscode': '200',
  'digest': 'F6LOMYMLJZ6456ORIXH2MEBAOGR7SMNC',
  'length': '8588'},
 {'urlkey': 'com,nytimes)/subscription',
  'timestamp': '20210102181748',
  'original': 'https://www.nytimes.com/subscription',
  'mimetype': 'text/html',
  'statuscode': '200',
  'digest': 'BKRZ3GVOGJXOVIS73NM5NBFATASSWSAM',
  'length': '11181'},
 {'urlkey': 'com,nytimes)/subscription',
  'timestamp': '20210103125837',
  'original': 'https://www.nytimes.com/subscription',
  'mimetype': 'text/html',
  'statuscode': '200',
  'digest': 'PJ4RMZMZMTVA6LD6NG24KZUHONDGIKUL',
  'length': '10697'},
 {'urlkey': 'com,nytimes)/subscription',
  'timestamp': '20210104031110',
  'original': 'https://www.nytimes.com/subscription',
  'mimetype': 'text/html',
  'statuscode': '200',
  'digest': 'N7STK37FCNKWOGD5SU443NVMJIQGCC4Y',
  'length': '10510'},
 {'urlke

In [20]:
len(snapshots) # number of snapshots for nyt over 5 years (2021 - 2026)

1769

In [21]:
from src.wayback_snapshot_fetcher import fetch_snapshot_price, sample_snapshots_across_range

sample = sample_snapshots_across_range(snapshots, n=8)
results = [fetch_snapshot_price(s) for s in sample]
results

[{'timestamp': '20210101012345',
  'wayback_url': 'https://web.archive.org/web/20210101012345/https://www.nytimes.com/subscription/',
  'status': 200,
  'prices_found': [],
  'num_prices': 0},
 {'timestamp': '20210812050645',
  'wayback_url': 'https://web.archive.org/web/20210812050645/https://www.nytimes.com/subscription',
  'status': None,
  'prices_found': None,
  'num_prices': 0,
  'error': "HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20210812050645/https://www.nytimes.com/subscription (Caused by ConnectTimeoutError(<HTTPSConnection(host='web.archive.org', port=443) at 0x273d6997f10>, 'Connection to web.archive.org timed out. (connect timeout=30)'))"},
 {'timestamp': '20220321003608',
  'wayback_url': 'https://web.archive.org/web/20220321003608/https://www.nytimes.com/subscription',
  'status': None,
  'prices_found': None,
  'num_prices': 0,
  'error': '429 Client Error: Too Many Requests for url: https://web.archive.org/web/202203210

In [22]:
fetch_snapshot_price(sample[1])

{'timestamp': '20210812050645',
 'wayback_url': 'https://web.archive.org/web/20210812050645/https://www.nytimes.com/subscription',
 'status': None,
 'prices_found': None,
 'num_prices': 0,
 'error': '429 Client Error: Too Many Requests for url: https://web.archive.org/web/20210812050645/https://www.nytimes.com/subscription'}

In [23]:
import inspect
print(inspect.getsource(fetch_snapshot_price))

def fetch_snapshot_price(snapshot, context_chars=60):
    """
    Fetches one archived Wayback snapshot and searches its HTML for a
    price pattern. Also captures surrounding text for each match so we
    can eyeball whether it's a real price or noise (like the JS
    backreference false positive found on the live NYT site).
    """
    timestamp = snapshot["timestamp"]
    original_url = snapshot["original"]
    wayback_url = f"https://web.archive.org/web/{timestamp}/{original_url}"

    try:
        response = requests.get(wayback_url, timeout=30)
        response.raise_for_status()

        matches_with_context = []
        for m in re.finditer(price_pattern, response.text):
            start = max(0, m.start() - context_chars)
            end = min(len(response.text), m.end() + context_chars)
            matches_with_context.append({
                "match": m.group(),
                "context": response.text[start:end]
            })

        return {
            "timestamp": ti

In [24]:
fetch_snapshot_price(sample[1])

{'timestamp': '20210812050645',
 'wayback_url': 'https://web.archive.org/web/20210812050645/https://www.nytimes.com/subscription',
 'status': None,
 'prices_found': None,
 'num_prices': 0,
 'error': "HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /web/20210812050645/https://www.nytimes.com/subscription (Caused by ConnectTimeoutError(<HTTPSConnection(host='web.archive.org', port=443) at 0x273d695b8d0>, 'Connection to web.archive.org timed out. (connect timeout=30)'))"}

In [26]:
fetch_snapshot_price


<function src.wayback_snapshot_fetcher.fetch_snapshot_price(snapshot, context_chars=60)>

In [27]:
# subscription detection
check_domain_full("www.wsj.com")
check_domain_full("www.theguardian.com")
check_domain_full("edition.cnn.com")

[{'domain': 'edition.cnn.com',
  'path': 'subscribe',
  'status': 200,
  'final_url': 'https://edition.cnn.com/subscription',
  'verdict': 'likely positive',
  'price_matches': [{'match': '$1.99',
    'context': '="true"\n>\n        Unlock two months of unlimited access for $1.99/month.\n</h2>\n\n  \n    \n    \n    \n\n<div class="subscription-c'},
   {'match': '$69.99',
    'context': 'ss="subscription-card-grouped-products__pricing-info-price">$69.99</span>\n                        <span class="subscription-ca'},
   {'match': '$1.99',
    'context': 'ss="subscription-card-grouped-products__pricing-info-price">$1.99</span>\n                        <span class="subscription-ca'},
   {'match': '$29.99',
    'context': 'ss="subscription-card-grouped-products__pricing-info-price">$29.99</span>\n                        <span class="subscription-ca'},
   {'match': '$3.99',
    'context': 'ss="subscription-card-grouped-products__pricing-info-price">$3.99</span>\n                        <spa

ReadTimeout: HTTPSConnectionPool(host='web.archive.org', port=443): Read timed out. (read timeout=60)